# Manager Agent — Code Guide

**Owner:** Jack
**Target file:** `agents/jack_manager.py`

> This notebook is documentation only — do not run it. It mirrors the actual agent code block by block and explains what each part does in plain English, so it's easier for the team (and future me) to understand this code inside out.

## What this agent does

The Manager is the **decision-maker** and the **bookender** of the pipeline. It runs at two moments:

- **The gate** — after Sabina scores the predictions, the Manager reads `evaluation_report.json` and decides, by a pure deterministic rule, whether to **retune** (send Nadi better hyperparameters and loop again) or **proceed** (accept the accuracy and move to explanations). When it retunes it writes `retune_request.json`; when it proceeds it writes `sample_for_explanation.csv` for Freddi.
- **The finalize** — once Freddi's `explanations.csv` is back, the Manager joins everything and writes `final_results.csv` + `final_report.json`.

**The key design principle: the gate is pure rules; the LLM only narrates.** The retune-vs-proceed decision is made by deterministic code you can unit-test. A Llama model is used *only* to write a human-readable rationale for the audit log — it can never change the decision. That separation is worth internalising: it's why this agent is trustworthy.

> **New to LangGraph?** Read `docs/langgraph_pipeline_guide.ipynb` (Part A) first — it explains state, nodes, reducers, conditional edges, and checkpointers, which this guide leans on throughout.


## 1. Module docstring & imports

The docstring states the contract up front: read Sabina's report, decide via a deterministic accuracy gate, write the retune request / sample / finals. *"LLM only writes the human-readable rationale — the gate itself is pure rules."*

The `try/except ModuleNotFoundError` import (same as every agent) makes the file work both as a package import in tests and as a direct script. `OUTPUT_DIR = "outputs"` is where every file this agent produces lands.


In [ ]:
"""Manager Agent (Jack) — orchestration loop + threshold gate + final output.

Reads Sabina's evaluation_report.json, decides retune-vs-proceed via a
deterministic accuracy gate, and (later steps) writes the retune request,
explanation sample, and final outputs. LLM (Llama via HF) only writes the
human-readable rationale — the gate itself is pure rules.

See docs/data_contracts.md (Handoffs 3, 3b, 4, 6).
"""

import json
import operator
import os
from typing import Annotated, TypedDict

# Works both as a package import (`import agents.jack_manager` in tests) and as a
# direct script (`uv run agents/jack_manager.py`, where `agents/` is on sys.path).
try:
    from agents.base import Agent
except ModuleNotFoundError:
    from base import Agent

OUTPUT_DIR = "outputs"

## 2. `ManagerState` — the shared clipboard (with reducers)

This is the state threaded through the Manager's own LangGraph. It's bigger than the pipeline state because the Manager genuinely needs to *remember* things across loop passes. Read it in the four groups the comments mark:

- **inputs / config** — set once at start (`evaluation_report`, `target_accuracy`, `max_iterations`).
- **loop progress** — plain fields, so returning them **overwrites** (latest-wins): `iteration`, `accuracy`, `final_action`, `decision`, `overrides`, `notes`.
- **convergence config** — `patience` and `min_delta`, the early-stopping knobs.
- **running history — the reducer fields.** This is the important part. `decision_log`, `accuracy_history`, and `tried_params` are each `Annotated[list, operator.add]`. That annotation means a node returns a **one-element list** and LangGraph **appends** it instead of overwriting (idea #3 from the LangGraph guide). This is *how the Manager remembers* what happened in earlier iterations — and that memory is the raw material for the convergence and adaptation decisions further down.

Contrast the two update styles deliberately: `accuracy` is a plain field (you only care about the current one), but `accuracy_history` is a reducer field (you need all of them to detect a plateau).


In [ ]:
class ManagerState(TypedDict):
    """Shared state threaded through the LangGraph. Nodes return partial
    updates to these keys; LangGraph merges them in (latest-wins), except
    `decision_log`, which appends via its reducer."""

    # --- inputs / config (set once at start) ---
    evaluation_report: dict      # parsed Sabina report for this iteration
    target_accuracy: float       # gate threshold, 0.60 per contract
    max_iterations: int          # iteration cap before forced proceed

    # --- loop progress (plain fields → merged, latest wins) ---
    iteration: int               # loop counter, starts at 1
    accuracy: float              # accuracy from the current report
    final_action: str            # "retune" | "proceed"
    decision: str                # "accept" | "override"
    overrides: dict              # fields Jack changed; {} if accept
    notes: str                   # rationale (filled by the LLM node, step 5)

    # --- convergence config (set once at start; see `decide`) ---
    patience: int                # how many recent iterations to watch for progress
    min_delta: float             # smallest accuracy gain that counts as "progress"

    # --- running history (reducer fields → APPENDED every iteration) ------------
    # A reducer field is merged with `operator.add` (list concatenation) instead of
    # being overwritten, so each node returns a *one-element list* and LangGraph
    # appends it. This is how the Manager remembers what happened in earlier
    # iterations — the raw material for the convergence and adaptation decisions.
    decision_log: Annotated[list, operator.add]      # human-readable note per step
    accuracy_history: Annotated[list, operator.add]  # one accuracy per iteration
    tried_params: Annotated[list, operator.add]      # retune params used per retune

    # --- I/O config (paths to contract files; optional, read via .get) ---
    predictions_path: str        # Nadi's predictions_test.csv (to sample / finalize)
    explanations_path: str       # Freddi's explanations.csv (present → finalize)
    sample_size: int             # rows for sample_for_explanation.csv (~300)

## 3. File-writing helpers

Two small helpers used by the terminal nodes:

- **`_write_json`** — dump a dict to `outputs/<name>`, creating the folder if needed.
- **`_write_decision`** — writes `decision.json`, Jack's audit record, **every iteration** (Handoff 3b). Note the comment: this file is *overwritten* each iteration, so once the loop moves on, `accuracy_history` (which is cumulative) is the only place the per-iteration trend survives on disk. That's a subtle but deliberate consequence of the reducer design in section 2.


In [ ]:
def _write_json(name: str, obj: dict) -> None:
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    with open(os.path.join(OUTPUT_DIR, name), "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)


def _write_decision(state: "ManagerState") -> None:
    """decision.json — Jack's record, written every iteration (Handoff 3b). The
    file is overwritten each iteration, so `accuracy_history` (cumulative through
    the current iteration) is the only place the per-iteration trend survives on
    disk once the loop moves on."""
    report = state["evaluation_report"]
    _write_json("decision.json", {
        "iteration": state["iteration"],
        "decision": state["decision"],
        "final_action": state["final_action"],
        "based_on_proposal": report.get("proposal", {}),
        "overrides": state.get("overrides", {}),
        "notes": state["notes"],
        "accuracy_history": state.get("accuracy_history", []),
    })

## 4. `_RETUNE_SCHEDULE` — the escalation ladder

When the gate decides to retune more than once, it can't just resend the same params — that would regenerate an identical classifier and waste an iteration. So we keep an **ordered escalation ladder**. Each rung pushes three levers further:

- **lower `threshold`** → fewer rows get forced into the `neutral` fallback class,
- **wider `max_length`** → longer headlines aren't truncated,
- **higher `boost_factor`** → Nadi's focus-label boost (on whatever Sabina flagged as weakest) hits harder.

The first retune trusts Sabina's proposal as-is; every retune *after* that walks down this list. All the tunable values live here — nothing downstream is hardcoded to them.


In [ ]:
# Ordered escalation schedule for the retune loop. The first retune re-uses
# Sabina's proposed params as-is; every retune AFTER that walks down this list
# (skipping anything already tried) so each attempt is genuinely different rather
# than a repeat. The levers: lower `threshold` so fewer rows get forced to the
# `neutral` fallback class, widen `max_length` so longer headlines aren't
# truncated, and raise `boost_factor` so Nadi's focus-label boost (applied to
# whatever Sabina flagged as weakest) has more effect each retune. Tune these
# values here — nothing downstream is hardcoded to them.
_RETUNE_SCHEDULE = [
    {"threshold": 0.45, "max_length": 128, "boost_factor": 1.25},
    {"threshold": 0.40, "max_length": 160, "boost_factor": 1.35},
    {"threshold": 0.35, "max_length": 192, "boost_factor": 1.45},
    {"threshold": 0.30, "max_length": 224, "boost_factor": 1.55},
    {"threshold": 0.25, "max_length": 256, "boost_factor": 1.65},
    {"threshold": 0.20, "max_length": 256, "boost_factor": 1.75},
]

## 5. `_next_params` — pick the next unused rung

Given the params we've already tried, return the first schedule rung we haven't (or the last rung once the ladder is exhausted — the iteration cap still bounds the loop, so a repeat here is safe).

The subtle bit is `_same`: it compares only the **keys the two dicts share**. Why? Sabina's proposal omits params she doesn't set (like `boost_factor`), and Nadi fills those from the same defaults the schedule starts at. So a rung that matches a tried set on *every shared key* would in practice regenerate an identical classifier — this comparison catches that and skips it.


In [ ]:
def _next_params(tried: list) -> dict:
    """Pick the next retune params: the first schedule entry not already tried,
    or the last entry once the schedule is exhausted (the iteration cap still
    bounds the loop, so returning a repeat here is safe).

    "Already tried" compares only the keys both dicts share: Sabina's proposal
    omits params she doesn't set (e.g. `boost_factor`), and Nadi fills those from
    the same defaults the schedule starts at — so a schedule entry that matches a
    tried set on every shared key would regenerate an identical classifier."""
    def _same(a: dict, b: dict) -> bool:
        shared = a.keys() & b.keys()
        return bool(shared) and all(a[k] == b[k] for k in shared)

    for params in _RETUNE_SCHEDULE:
        if not any(_same(params, t) for t in tried):
            return params
    return _RETUNE_SCHEDULE[-1]

## 6. `_converged` — has retuning stopped paying off?

This is the early-stopping rule, and it's the reason the loop doesn't burn all five iterations chasing accuracy that isn't coming.

The rule in plain English: once we have more than `patience` accuracies recorded, take the **best of the last `patience` iterations** and compare it to the **best of everything before**. If the recent window failed to beat the earlier best by at least `min_delta`, we've plateaued → converged → proceed. Until we have enough history, it returns `False` and lets the loop keep exploring. `history` includes the current iteration's accuracy.


In [ ]:
def _converged(history: list, patience: int, min_delta: float) -> bool:
    """True when retuning has stopped paying off, so we should proceed instead of
    burning the rest of the iteration budget.

    Rule: once we have more than `patience` accuracies, compare the best of the
    last `patience` iterations against the best of everything before them. If the
    recent window failed to beat the earlier best by at least `min_delta`, the
    loop has plateaued. `history` includes the current iteration's accuracy.
    """
    if len(history) <= patience:
        return False  # not enough history yet — let the loop keep exploring
    recent_best = max(history[-patience:])
    earlier_best = max(history[:-patience])
    return recent_best - earlier_best < min_delta

## 7. `decide` — the gate node (the brain)

This is the most important function in the file and the first node the graph runs. It's pure, deterministic, and returns **only the keys it changed** (LangGraph merges them in). Walk it in three parts:

**(a) Count the iteration.** Bump the counter *only* if we haven't already proceeded — the finalize pass must not count as a new iteration.

**(b) The three independent stop conditions.** The gate proceeds if **any** of these is true:
- `cleared` — accuracy met the target (good enough),
- `cap_hit` — we're out of iteration budget,
- `converged` — progress has stalled (section 6).

Otherwise it retunes. The `why` string records *which* reason fired, for the audit note.

**(c) Build the update.** Note which keys are plain (`accuracy`, `notes` → overwrite) vs reducer (`decision_log`, `accuracy_history` → appended, each returned as a one-element list). This is section 2's reducer design in action.

**On retune:** the first retune *accepts* Sabina's proposal as-is; every retune after that *overrides* with the next ladder rung (a repeat proposal has already failed once). Either way `tried_params` records what was actually used.

**On proceed:** it only marks an *override* when the gate overrules Sabina (e.g. she said retune but the cap/convergence forced proceed); otherwise it's an accept.


In [ ]:
def decide(state: ManagerState) -> dict:
    """Threshold gate (deterministic). Decides retune vs. proceed, and when
    retuning, chooses the next hyperparameters from history. Returns only the
    keys it changed (LangGraph merges them into the running state).
    """
    report = state["evaluation_report"]
    accuracy = report["accuracy"]
    # Count retune cycles only: once we've proceeded, the finalize pass is not a
    # new iteration, so don't bump the counter past convergence.
    iteration = state.get("iteration", 0)
    if state.get("final_action") != "proceed":
        iteration += 1

    # Full accuracy trend INCLUDING this iteration — the input to convergence.
    history = state.get("accuracy_history", []) + [accuracy]
    patience = state.get("patience", 2)
    min_delta = state.get("min_delta", 0.01)

    # Three independent reasons to stop retuning and move on.
    cleared = accuracy >= state["target_accuracy"]       # good enough
    cap_hit = iteration >= state["max_iterations"]        # out of budget
    converged = _converged(history, patience, min_delta)  # progress has stalled
    final_action = "proceed" if (cleared or cap_hit or converged) else "retune"

    if cleared:
        why = "cleared target"
    elif converged:
        why = "converged (no improvement), proceeding"
    elif cap_hit:
        why = "cap hit, forcing proceed"
    else:
        why = "below target, retuning"
    note = (f"iteration {iteration}: accuracy {accuracy:.2f} vs target "
            f"{state['target_accuracy']:.2f} — {why}")

    out = {
        "iteration": iteration,          # saved → next invocation resumes from here
        "accuracy": accuracy,
        "final_action": final_action,
        "notes": note,                   # plain field → overwrites
        "decision_log": [note],          # reducer field → appended
        "accuracy_history": [accuracy],  # reducer field → appended
    }

    if final_action == "retune":
        # First retune trusts Sabina's proposal as-is (accept); every retune after
        # that adapts the params (override), because a repeat proposal has already
        # failed once. `tried_params` records what we actually used either way.
        proposal = report.get("proposal", {})
        tried = state.get("tried_params", [])
        if not tried:
            used = proposal.get("suggested_params", {})
            out["decision"], out["overrides"] = "accept", {}
        else:
            used = _next_params(tried)
            out["decision"] = "override"
            out["overrides"] = {"suggested_params": used,
                                "focus_labels": proposal.get("focus_labels", [])}
        out["tried_params"] = [used]
    else:
        # Proceeding. Override only when the gate overrules Sabina's recommendation
        # (e.g. she says retune but the cap/convergence forces proceed); else accept.
        recommended = report.get("proposal", {}).get("recommended_action")
        if recommended is not None and recommended != final_action:
            out["decision"], out["overrides"] = "override", {"final_action": final_action}
        else:
            out["decision"], out["overrides"] = "accept", {}

    return out

## 8. `route_after_decide` — the router

This is the conditional-edge router (idea #5 from the LangGraph guide). It reads decisions the nodes already made and returns one of three labels:

- `"retune"` → go write the retune request,
- `"finalize"` → Freddi's explanations are already back, so finish,
- `"sample"` → proceeding but explanations aren't back yet, so draw the sample and wait.

It makes no decisions of its own — it just reads `final_action` and whether `explanations_path` is set, and picks the branch.


In [ ]:
def route_after_decide(state: ManagerState) -> str:
    """Router: retune, or — on proceed — finalize if Freddi's explanations are
    back, else sample and wait. Reads decisions the nodes already made."""
    if state["final_action"] == "retune":
        return "retune"
    return "finalize" if state.get("explanations_path") else "sample"

## 9. `write_retune` — the retune branch (terminal node)

Writes `retune_request.json` (Handoff 3b), the approved proposal Nadi will read. It merges Sabina's proposal with any overrides the gate applied — `focus_labels` and `suggested_params` come from the overrides if present, else the proposal. It also calls `_write_decision` for the audit trail.

This node is **terminal** (it goes straight to `END`): the Manager has handed off and now waits for Nadi + Sabina to produce the next report, which arrives as the *next* `.run()` call on the same instance. The returned `decision_log` entry is appended via the reducer.


In [ ]:
def write_retune(state: ManagerState) -> dict:
    """retune_request.json (Handoff 3b) — approved proposal for Nadi. Terminal:
    Manager hands off and waits for Nadi+Sabina to produce the next report."""
    report = state["evaluation_report"]
    proposal = report.get("proposal", {})
    overrides = state.get("overrides", {})
    _write_decision(state)
    _write_json("retune_request.json", {
        "iteration": state["iteration"],
        "reason": proposal.get("reason", state["notes"]),
        "current_accuracy": state["accuracy"],
        "target_accuracy": state["target_accuracy"],
        "focus_labels": overrides.get("focus_labels", proposal.get("focus_labels", [])),
        "misclassified_ids": report.get("misclassified_ids", []),
        "suggested_params": {**proposal.get("suggested_params", {}),
                             **overrides.get("suggested_params", {})},
    })
    return {"decision_log": [f"iteration {state['iteration']}: wrote retune_request.json"]}

## 10. `proceed` — the sample branch (terminal node)

Writes `sample_for_explanation.csv` (Handoff 4) for Freddi, drawn from Nadi's `predictions_test.csv`. It selects the contract columns, renames `label → actual_label`, and subsamples **only when needed** (`n < len(sample)`) with a fixed `random_state=42` so the sample is representative *and* reproducible. Also terminal — hands off to Freddi and waits for `explanations.csv`.


In [ ]:
def proceed(state: ManagerState) -> dict:
    """sample_for_explanation.csv (Handoff 4) — drawn from predictions_test.csv.
    Terminal: hands off to Freddi and waits for explanations.csv."""
    import pandas as pd

    preds = pd.read_csv(state.get("predictions_path", "mock_data/predictions_test.csv"))
    sample = (preds[["article_id", "article_title", "predicted_label", "label",
                     "confidence", "prob_up", "prob_down", "prob_neutral"]]
              .rename(columns={"label": "actual_label"}))
    n = min(len(sample), state.get("sample_size", 300))
    if n < len(sample):                                   # only subsample when needed
        sample = sample.sample(n=n, random_state=42)      # representative + reproducible
    _write_decision(state)
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    sample.to_csv(os.path.join(OUTPUT_DIR, "sample_for_explanation.csv"), index=False)
    return {"decision_log": [f"iteration {state['iteration']}: wrote sample_for_explanation.csv ({n} rows)"]}

## 11. `finalize` — the finalize branch (terminal node)

The last node in a successful run. Once `explanations.csv` is back, it joins predictions ↔ explanations on `article_id` (left join, so every prediction survives) and writes the two Handoff-6 deliverables:

- **`final_results.csv`** — the full table, with `explanation` nulls filled to empty string and `manual_score` as a nullable `Int64` (int, NA → empty per the contract).
- **`final_report.json`** — the run summary: final accuracy, loop iterations, per-class accuracy, test-set size, and counts of explanations generated / manually scored.


In [ ]:
def finalize(state: ManagerState) -> dict:
    """final_results.csv + final_report.json (Handoff 6) — once explanations.csv
    is back from Freddi. Joins predictions to explanations and writes the finals."""
    import pandas as pd

    preds = pd.read_csv(state.get("predictions_path", "mock_data/predictions_test.csv"))
    expl = pd.read_csv(state["explanations_path"])[["article_id", "explanation", "manual_score"]]
    final = preds.merge(expl, on="article_id", how="left")[[
        "article_id", "date", "ticker", "article_title", "price_t", "price_t1",
        "pct_change", "label", "predicted_label", "confidence", "explanation", "manual_score"]]
    final["explanation"] = final["explanation"].fillna("")          # null → empty string
    final["manual_score"] = final["manual_score"].astype("Int64")   # int, NA → empty

    report = state["evaluation_report"]
    _write_decision(state)
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    final.to_csv(os.path.join(OUTPUT_DIR, "final_results.csv"), index=False)
    _write_json("final_report.json", {
        "final_accuracy": report.get("accuracy"),
        "loop_iterations": state["iteration"],
        "class_accuracy": report.get("class_accuracy", {}),
        "test_set_size": int(len(preds)),
        "explanations_generated": int((final["explanation"] != "").sum()),
        "manually_scored": int(final["manual_score"].notna().sum()),
    })
    return {"decision_log": [f"iteration {state['iteration']}: wrote final_results.csv + final_report.json"]}

## 12. The LLM rationale prompt — narration, not decision

Now the LLM part — and remember, it only *narrates*. `RATIONALE_SYSTEM_PROMPT` is engineered with the lecture's prompt components, and the comment maps them: Persona (line 1), Sections (`###`), CAPITALS (the determinism boundary), Output spec (plain prose), Example (one-shot).

The most important lines are the constraints: the decision *has already been made* by the deterministic gate — *"You did NOT make it and you CANNOT change it."* The model writes a 2–3 sentence justification for the human audit log and nothing more. This is the prompt-level guarantee that the LLM can't override the rules.


In [ ]:
LLAMA_MODEL = "meta-llama/Llama-3.1-8B-Instruct"

# Engineered with the lecture's components: Persona (line 1), Sections (### ),
# CAPITALS (the determinism boundary), Output (plain prose), Example (one-shot).
# Thinking/CoT deliberately omitted — the task is too simple to need it.
RATIONALE_SYSTEM_PROMPT = """You are the MANAGER AGENT of an ML pipeline — a precise, \
factual orchestrator who explains decisions for a human audit log.

### CONTEXT ###
A retune-vs-proceed decision has ALREADY been made by a deterministic accuracy gate.
You did NOT make it and you CANNOT change it.

### YOUR TASK ###
Write a 2-3 sentence rationale explaining WHY the decision is reasonable, grounded in
the accuracy, the target, and the evaluator's proposal.

### CONSTRAINTS ###
- DO NOT dispute, second-guess, or suggest changing the decision.
- Output PLAIN PROSE ONLY — no code, JSON, markdown, lists, or preamble.
- Be factual and concise. No marketing tone.

### EXAMPLE ###
Input  — Decision: proceed at iteration 2. Accuracy 0.67 vs target 0.60. Proposal: proceed.
Output — Test accuracy of 0.67 clears the 0.60 target, so the gate proceeds to the \
explanation stage. The evaluator agreed; though the neutral class remains weakest, the \
iteration budget favours moving forward.
"""

## 13. `_llama_rationale` — call Llama, but never break the graph

Asks Llama (via the HuggingFace `InferenceClient`) for the rationale — with **two layers of graceful fallback**, because a missing token or a flaky API must never abort a run whose *decision is already made and needs to be written*:

1. **No `HF_TOKEN`** → skip the call entirely and return the gate's deterministic note with a `(LLM skipped)` marker. This is what lets offline `mock_data` tests run.
2. **Any HF failure** (rate limit, gated-model 403, timeout, malformed response) → caught by the broad `except`, returns the deterministic note with `(LLM failed: ...)`.

Either way the graph keeps moving and the audit log still gets *a* rationale. This is the [[silent-failure]] concern handled correctly: the fallback is visible (marked in the note), not hidden.


In [ ]:
def _llama_rationale(state: ManagerState) -> str:
    """Ask Llama for a short rationale. Falls back to the gate's deterministic
    note when HF_TOKEN is unset, so offline mock_data tests still run."""
    token = os.environ.get("HF_TOKEN")
    if not token:
        return state["notes"] + " (LLM skipped: no HF_TOKEN)"

    from huggingface_hub import InferenceClient

    proposal = state["evaluation_report"].get("proposal", {})
    # Any HF failure (rate limit, gated-model 403, timeout, malformed response)
    # must not abort the graph — the gate's decision still needs to be written.
    # Fall back to the deterministic note, mirroring the no-token branch above.
    try:
        client = InferenceClient(model=LLAMA_MODEL, token=token)
        resp = client.chat_completion(
            messages=[
                {"role": "system", "content": RATIONALE_SYSTEM_PROMPT},
                {"role": "user", "content": (
                    f"Decision: {state['final_action']} at iteration {state['iteration']}. "
                    f"Test accuracy {state['accuracy']:.2f} vs target "
                    f"{state['target_accuracy']:.2f}. Evaluator proposal: {proposal}.")},
            ],
            max_tokens=160,
            temperature=0.3,
        )
        return resp.choices[0].message.content.strip()
    except Exception as e:
        return state["notes"] + f" (LLM failed: {e})"

## 14. `rationale` — the LLM node

A one-liner node that writes `notes` **only**. The comment is a load-bearing reminder: this node *never* touches `final_action` or any control-flow key. The LLM's output is quarantined to a single human-readable field. That's the whole safety story in one function.


In [ ]:
def rationale(state: ManagerState) -> dict:
    """LLM node: writes `notes` ONLY. Never touches final_action (control flow)."""
    return {"notes": _llama_rationale(state)}

## 15. `build_graph` — wiring the Manager's graph

Now assemble the nodes into the Manager's LangGraph:

```
START → decide → rationale ─(retune)→ write_retune → END
                          ├(sample)→ proceed → END
                          └(finalize)→ finalize → END
```

Read the edges: `decide` always flows to `rationale` (**gate first, then explain** — the decision is locked before the LLM ever speaks), then `add_conditional_edges` uses `route_after_decide` to pick one of the three terminal nodes. Every branch ends at `END`. Unlike the pipeline graph, this one has **no cycle** — the loop lives in the *pipeline* graph (or in `main.py`), which calls this Manager once per pass.


In [ ]:
def build_graph(checkpointer):
    from langgraph.graph import StateGraph, START, END

    b = StateGraph(ManagerState)
    b.add_node("decide", decide)
    b.add_node("rationale", rationale)
    b.add_node("write_retune", write_retune)
    b.add_node("proceed", proceed)
    b.add_node("finalize", finalize)
    b.add_edge(START, "decide")
    b.add_edge("decide", "rationale")       # gate first, then explain
    b.add_conditional_edges(
        "rationale", route_after_decide,
        {"retune": "write_retune", "sample": "proceed", "finalize": "finalize"},
    )
    b.add_edge("write_retune", END)
    b.add_edge("proceed", END)
    b.add_edge("finalize", END)
    return b.compile(checkpointer=checkpointer)

## 16. `ManagerAgent` — the public `.run()` wrapper

The team convention (see `agents/base.py`): subclass `Agent`, implement `build_graph` (delegates to the module function) and `run`. Construct once, then feed contract file paths in.

The `__init__` stashes `_defaults` — the config that's set once and merged into every run's state, while the iteration counter and history reducers **accumulate across runs via the checkpointer** (idea #6). That's the mechanism that makes a single `ManagerAgent` instance behave like a stateful loop controller across repeated `.run()` calls.

`run(evaluation_report, explanations=None)` shapes the state from file paths: passing just `evaluation_report` means retune-or-sample; adding `explanations` means finalize. It reads the report JSON, builds the state, and calls the base `_invoke`.


In [ ]:
class ManagerAgent(Agent):
    """Manager (Jack) behind the shared `.run()` interface. Construct once, then
    feed contract file paths in: pass `evaluation_report` to retune or sample,
    add `explanations` to finalize. Outputs land in `OUTPUT_DIR` per the contract.

        mgr = ManagerAgent()
        mgr.run(evaluation_report="outputs/evaluation_report.json")
        mgr.run(evaluation_report="outputs/evaluation_report.json",
                explanations="outputs/explanations.csv")
    """

    def __init__(self, *, target_accuracy=0.60, max_iterations=5,
                 patience=2, min_delta=0.01,
                 predictions_path="mock_data/predictions_test.csv",
                 sample_size=300, checkpointer=None, thread_id="manager"):
        # Set once and merged into every run's state; the iteration counter and
        # the history reducers accumulate across runs via the checkpointer.
        # `patience`/`min_delta` tune early-stopping: proceed once accuracy hasn't
        # gained `min_delta` over the best of the last `patience` iterations.
        self._defaults = {
            "target_accuracy": target_accuracy,
            "max_iterations": max_iterations,
            "patience": patience,
            "min_delta": min_delta,
            "predictions_path": predictions_path,
            "sample_size": sample_size,
        }
        super().__init__(checkpointer=checkpointer, thread_id=thread_id)

    def build_graph(self, checkpointer):
        return build_graph(checkpointer)

    def run(self, evaluation_report: str, explanations: str | None = None) -> dict:
        """`evaluation_report`: path to Sabina's report JSON. `explanations`:
        path to Freddi's explanations.csv — present means finalize, absent means
        retune-or-sample."""
        with open(evaluation_report, encoding="utf-8") as f:
            report = json.load(f)
        state = {**self._defaults, "evaluation_report": report}
        if explanations is not None:
            state["explanations_path"] = explanations
        return self._invoke(state)

## 17. Command-line entry point (TESTING)

Only runs when the file is executed directly. It drives the whole loop through the public `.run()` API three times on **one instance** — so the in-memory checkpointer carries iteration state across all three calls, hitting each branch in turn:

- **R1** — a synthesised below-target report → **retune** branch (`write_retune`).
- **R2** — the shipped proceed report → **sample** branch (`proceed`).
- **R3** — same report *plus* explanations → **finalize** branch.

It prints the iteration/action for each, then lists what landed in `outputs/`. A quick, dependency-free way to watch all three branches fire without the rest of the pipeline.


In [ ]:
if __name__ == "__main__":
    import tempfile

    # Drive the whole loop through the public ManagerAgent.run() file API. One
    # instance keeps iteration state across the three runs via its in-memory
    # checkpointer (no sqlite cleanup needed — fresh process, fresh state).
    mgr = ManagerAgent(thread_id="demo")

    proceed_path = "mock_data/evaluation_report.json"      # ships a proceed report
    retune_report = {
        "accuracy": 0.54, "below_threshold": True,
        "class_accuracy": {"up": 0.60, "down": 0.40, "neutral": 0.30},
        "misclassified_ids": ["FNSPID_00006", "FNSPID_00010"],
        "proposal": {
            "recommended_action": "retune",
            "reason": "accuracy 0.54 below target 0.60; down/neutral weakest",
            "focus_labels": ["down", "neutral"],
            "suggested_params": {"threshold": 0.5, "max_length": 128},
            "code_notes": "threshold hardcoded at 0.5 in classifier.py"},
    }
    # No retune report ships in mock_data/, so materialise one to a temp file —
    # the API takes a path, so the demo feeds it a path.
    with tempfile.NamedTemporaryFile("w", suffix=".json", delete=False) as f:
        json.dump(retune_report, f)
        retune_path = f.name

    r1 = mgr.run(evaluation_report=retune_path)
    print(f"R1 → it={r1['iteration']} action={r1['final_action']:7s} → retune branch")
    r2 = mgr.run(evaluation_report=proceed_path)
    print(f"R2 → it={r2['iteration']} action={r2['final_action']:7s} → sample branch")
    r3 = mgr.run(evaluation_report=proceed_path, explanations="mock_data/explanations.csv")
    print(f"R3 → it={r3['iteration']} action={r3['final_action']:7s} → finalize branch")

    os.remove(retune_path)
    print("\noutputs/ now holds:", sorted(os.listdir(OUTPUT_DIR)))